# 🧠 Flow Matching Tutorial – Fill in the Code

This version is designed as an exercise. You will complete sections of the model and training logic. Read the prompts and implement the missing pieces.

In [2]:
import torch
import torch.nn as nn
import numpy as np
from torch import Tensor
from matplotlib import pyplot as plt

## Define the Flow Model
Fill in the missing parts of the model definition.

In [6]:
class Flow(nn.Module):
    """
    PyTorch employs object oriented programming with class inheritance 
    
    Here, we define a class, Flow, that inherits all classes, attributes, and methods of the parent class nn.Module, built by PyTorch
    """
    def __init__(self, dim: int = 2, h: int = 64): # initialize attributes
        super().__init__() # call the constructor of the parent class

        # define a neural network with 3 hidden ELU layer
        self.net = nn.Sequential(
            nn.Linear(dim + 1, h), # first layer defined here with activation function, dim + 1 for xt (2-dimensional) and t
            nn.ELU(), 
            # TODO: define second layer here (hint: input size is the output size of the previous layer) 

            # TODO: define third layer here 

        )

    # define the forward pass through the neural  network
    def forward(self, t: Tensor, xt: Tensor) -> Tensor:
        combined_input = torch.cat((t, xt), -1)  # concatenate time and input vector
        return self.net(combined_input)

    # perform generic midpoint intergration for sampling
    def step(self, xt: Tensor, t_start: Tensor, t_end: Tensor) -> Tensor:
        # TODO: expand t_start to match batch size
        t_start = ___

        # TODO: Compute midpoint time
        midpoint_time = ___

        # compute velocity at t_start
        intermediate_value = self(xt=xt, t=t_start)

        # TODO: Move xt halfway using the intermediate value
        updated_xt = ___

        # xompute velocity at midpoint
        midpoint_output = self(t=midpoint_time, xt=updated_xt)

        # return full update using midpoint velocity
        return xt + (t_end - t_start) * midpoint_output



## Generate Training Data

In [ ]:
def sample_from():
    # Sample a checkerboard distribution for x1 and a Gaussian for x0
    rv1 = torch.rand(10000) * 4 - 2
    rv2 = torch.rand(10000) * 4 - 2
    mask = (torch.floor(rv1) + torch.floor(rv2)) % 2 == 0
    x1 = torch.stack((rv1[mask], rv2[mask]), dim=1)
    x0 = torch.randn_like(x1)
    return x0, x1

## Train the Model
Complete the training loop to minimize the MSE between predicted and target velocity.

In [ ]:
def train(flow, optimizer, loss_fn, epochs=10000):
    for _ in range(epochs):
        # TODO: sample x0, x1 using sample_from(...)
        x0, x1 = ___
        t = torch.rand(len(x1), 1) # t is the same length as batch size

        # TODO: generate xt for each t using optimal transport scheme (linear interpolation between x1 and x0)
        xt = ___ 

        # TODO: define the true velocity field using the optimal transport framework 
        v_true = ___

        # compute loss between predicted velocity and true velcocity
        loss = loss_fn(flow(t=t, xt=xt), v_true) 

        # TODO: backpropogate (see ChatGPT or tutorial online)

    print(f"Loss: {loss.item()}")


### Initialize Model and Start Training

In [ ]:
# TODO: initialize and train the model (see ChatGPT or tutorial online)
flow = Flow() # create instance of class Flow
optimizer = ___ # define optimizer (see ChatGPT or tutorial online)
loss_fn = ___ # define loss functiona (mean squared error)

# call train function train(...)


## Visualize the Transformation

In [ ]:
x = torch.randn(3000, 2) # generate random x 
n_steps = 8 # define steps between 0 and 1

fig, axes = plt.subplots(1, n_steps + 1, figsize=(30, 4), sharex=True, sharey=True)
time_steps = torch.linspace(0, 1.0, n_steps + 1)

axes[0].scatter(x.detach()[:, 0], x.detach()[:, 1], s=10, alpha=0.50)
axes[0].set_title(f't = {time_steps[0]:.2f}')
axes[0].set_xlim(-2.0, 2.0)
axes[0].set_ylim(-2.0, 2.0)

for i in range(n_steps):
    # TODO: define xt at any t using flow.step(...)
    x = ___
    axes[i + 1].scatter(x.detach()[:, 0], x.detach()[:, 1], s=10, alpha=0.50)
    axes[i + 1].set_title(f't = {time_steps[i + 1]:.2f}')
plt.show()


## Plot the Ground Truth Checkerboard Distribution

In [ ]:
rv1 = torch.rand(10000) * 4 - 2
rv2 = torch.rand(10000) * 4 - 2
mask = (torch.floor(rv1) + torch.floor(rv2)) % 2 == 0
x1 = torch.stack((rv1[mask], rv2[mask]), dim=1)
x1_np = x1.numpy()
plt.figure(figsize=(4, 4))
plt.scatter(x1_np[:, 0], x1_np[:, 1], s=10, alpha=0.50)
plt.title("Checkerboard Distribution")
plt.xlabel("X1")
plt.ylabel("X2")
plt.xlim(-2, 2)
plt.ylim(-2, 2)
plt.show()